# 03. SQL Analysis

SQL analysis is used to answer business questions by querying the cleaned food delivery dataset.

### Business Objective

Identify operational factors associated with delivery delays and answer key business questions using SQL.

### Key Analysis Areas

- Overall delivery performance
- Traffic and weather impact
- Distance and delivery time
- Vehicle and delivery partner performance
- Multiple deliveries
- City-level performance

In [5]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/cleaned/food_delivery_cleaned.csv")

df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,1900-01-01 11:30:00,1900-01-01 11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,1900-01-01 19:45:00,1900-01-01 19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,1900-01-01 08:30:00,1900-01-01 08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,1900-01-01 18:00:00,1900-01-01 18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,1900-01-01 13:30:00,1900-01-01 13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30


In [6]:
df.shape

(45593, 20)

In [7]:
conn = sqlite3.connect("../data/food_delivery.db")

df.to_sql(
    "food_delivery",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded into SQLite successfully.")

Data loaded into SQLite successfully.


In [8]:
query = """
SELECT *
FROM food_delivery
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,1900-01-01 11:30:00,1900-01-01 11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,1900-01-01 19:45:00,1900-01-01 19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,1900-01-01 08:30:00,1900-01-01 08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,1900-01-01 18:00:00,1900-01-01 18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,1900-01-01 13:30:00,1900-01-01 13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30


## 1. Overall Delivery Performance

Establish the overall delivery performance using SQL before investigating the factors that cause delays.

### Key Business Questions

- How many deliveries are in the dataset?
- What is the average delivery time?
- What are the fastest and slowest delivery times?
- What is the median delivery time?

In [9]:
query = """
SELECT
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time,
    MIN("Time_taken(min)") AS minimum_delivery_time,
    MAX("Time_taken(min)") AS maximum_delivery_time
FROM food_delivery;
"""

pd.read_sql_query(query, conn)

,total_deliveries,average_delivery_time,minimum_delivery_time,maximum_delivery_time
0,45593,26.29,10,54


In [10]:
query = """
SELECT
    AVG("Time_taken(min)") AS median_delivery_time
FROM (
    SELECT
        "Time_taken(min)",
        ROW_NUMBER() OVER (ORDER BY "Time_taken(min)") AS row_num,
        COUNT(*) OVER () AS total_rows
    FROM food_delivery
)
WHERE row_num IN (
    (total_rows + 1) / 2,
    (total_rows + 2) / 2
);
"""

pd.read_sql_query(query, conn)

,median_delivery_time
0,26.0


### Finding

The dataset contains **45,593 deliveries** with an average delivery time of **26.29 minutes**. The fastest delivery was completed in **10 minutes**, while the slowest took **54 minutes**.

This establishes the overall delivery-performance baseline and shows that delivery times vary considerably across orders.

**Business implication:** The overall average provides a baseline against which traffic, weather, workload, distance, vehicle, and delivery partner performance can be compared.

### Question 2: Late Delivery Analysis

What percentage of deliveries exceed the 30-minute delivery threshold?

In [11]:
query = """
SELECT
    CASE
        WHEN "Time_taken(min)" > 30 THEN 'Late'
        ELSE 'On Time'
    END AS delivery_status,
    COUNT(*) AS total_deliveries,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM food_delivery),
        2
    ) AS percentage
FROM food_delivery
GROUP BY delivery_status
ORDER BY total_deliveries DESC;
"""

pd.read_sql_query(query, conn)

,delivery_status,total_deliveries,percentage
0,On Time,31987,70.16
1,Late,13606,29.84


### Finding

The analysis shows that **31,987 deliveries (70.16%) were completed within 30 minutes**, while **13,606 deliveries (29.84%) were late**.

This means nearly **3 out of every 10 deliveries exceeded the 30-minute threshold**, indicating that delivery delays represent a significant operational issue.

**Business implication:** Reducing the late-delivery rate should be a key operational priority, with further analysis focused on identifying the traffic, weather, workload, distance, and vehicle conditions associated with delays.

## 3. Traffic Impact on Delivery Time

Analyze how traffic conditions affect delivery time and the likelihood of late deliveries using SQL.

### Key Business Questions

- Which traffic condition has the highest average delivery time?
- Which traffic condition has the highest late-delivery rate?
- How does delivery performance change from Low to Jam traffic?

In [12]:
query = """
SELECT
    "Road_traffic_density" AS traffic_condition,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time,
    MIN("Time_taken(min)") AS minimum_delivery_time,
    MAX("Time_taken(min)") AS maximum_delivery_time
FROM food_delivery
GROUP BY "Road_traffic_density"
ORDER BY average_delivery_time DESC;
"""

pd.read_sql_query(query, conn)

,traffic_condition,total_deliveries,average_delivery_time,minimum_delivery_time,maximum_delivery_time
0,Jam,14143,31.18,10,54
1,High,4425,27.24,10,54
2,Medium,10947,26.70,10,49
3,NaN,601,26.54,10,54
4,Low,15477,21.27,10,44


In [13]:
query = """
SELECT
    "Road_traffic_density" AS traffic_condition,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
GROUP BY "Road_traffic_density"
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,traffic_condition,total_deliveries,late_deliveries,late_delivery_rate
0,Jam,14143,7162,50.64
1,High,4425,1485,33.56
2,Medium,10947,3555,32.47
3,NaN,601,186,30.95
4,Low,15477,1218,7.87


### Finding

Traffic density has a clear impact on delivery performance. **Jam traffic** has the highest average delivery time at **31.18 minutes** and a **50.64% late-delivery rate**, compared with only **21.27 minutes** and **7.87%** during Low traffic.

This represents a difference of approximately **9.91 minutes** in average delivery time between Jam and Low traffic.

**Business implication:** Traffic conditions should be incorporated into ETA planning and route optimization. Additional operational controls may also be required during Jam traffic to reduce late deliveries.

## 4. Weather Impact on Delivery Time

Analyze how different weather conditions affect delivery time and late deliveries using SQL.

### Key Business Questions

- Which weather condition has the highest average delivery time?
- Which weather condition has the highest late-delivery rate?
- How does delivery performance vary across weather conditions?

In [14]:
query = """
SELECT
    "Weatherconditions" AS weather_condition,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time,
    MIN("Time_taken(min)") AS minimum_delivery_time,
    MAX("Time_taken(min)") AS maximum_delivery_time
FROM food_delivery
GROUP BY "Weatherconditions"
ORDER BY average_delivery_time DESC;
"""

pd.read_sql_query(query, conn)

,weather_condition,total_deliveries,average_delivery_time,minimum_delivery_time,maximum_delivery_time
0,conditions Fog,7654,28.92,10,54
1,conditions Cloudy,7536,28.92,10,54
2,None,616,26.55,10,54
3,conditions Windy,7422,26.12,10,54
4,conditions Sandstorms,7495,25.88,10,54
5,conditions Stormy,7586,25.87,10,54
6,conditions Sunny,7284,21.86,10,54


In [15]:
query = """
SELECT
    "Weatherconditions" AS weather_condition,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
GROUP BY "Weatherconditions"
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,weather_condition,total_deliveries,late_deliveries,late_delivery_rate
0,conditions Fog,7654,3352,43.79
1,conditions Cloudy,7536,3299,43.78
2,None,616,190,30.84
3,conditions Windy,7422,1971,26.56
4,conditions Sandstorms,7495,1945,25.95
5,conditions Stormy,7586,1937,25.53
6,conditions Sunny,7284,912,12.52


### Finding

Weather conditions are strongly associated with delivery performance. **Fog and Cloudy conditions** have the highest average delivery time at **28.92 minutes** and late-delivery rates of approximately **43.8%**.

In contrast, **Sunny conditions** have an average delivery time of **21.86 minutes** and a late-delivery rate of only **12.52%**.

**Business implication:** Weather conditions should be considered when planning ETAs and delivery capacity, particularly during Fog and Cloudy conditions.

## 5. Weather × Traffic Impact

Analyze how weather and traffic conditions interact to affect delivery performance using SQL.

### Key Business Questions

- Which weather and traffic combination has the highest average delivery time?
- Which combination has the highest late-delivery rate?
- How does poor weather combined with heavy traffic affect deliveries?
- Which combinations should be considered high-risk operational conditions?

In [16]:
query = """
SELECT
    "Weatherconditions" AS weather_condition,
    "Road_traffic_density" AS traffic_condition,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time
FROM food_delivery
GROUP BY
    "Weatherconditions",
    "Road_traffic_density"
ORDER BY average_delivery_time DESC;
"""

pd.read_sql_query(query, conn)

,weather_condition,traffic_condition,total_deliveries,average_delivery_time
0,None,High,1,49.00
1,conditions Fog,Jam,2429,36.81
2,conditions Cloudy,Jam,2349,36.69
3,None,Jam,3,35.33
4,conditions Windy,Jam,2351,30.22
5,conditions Sandstorms,Jam,2399,30.02
6,conditions Stormy,Jam,2323,29.85
7,conditions Cloudy,High,744,28.94
8,conditions Cloudy,Medium,1838,28.48
9,conditions Fog,High,776,28.43


In [17]:
query = """
SELECT
    "Weatherconditions" AS weather_condition,
    "Road_traffic_density" AS traffic_condition,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
GROUP BY
    "Weatherconditions",
    "Road_traffic_density"
HAVING COUNT(*) >= 100
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,weather_condition,traffic_condition,total_deliveries,late_deliveries,late_delivery_rate
0,conditions Fog,Jam,2429,1848,76.08
1,conditions Cloudy,Jam,2349,1758,74.84
2,conditions Cloudy,Medium,1838,888,48.31
3,conditions Fog,Medium,1852,850,45.90
4,conditions Windy,Jam,2351,1071,45.56
5,conditions Sandstorms,Jam,2399,1068,44.52
6,conditions Stormy,Jam,2323,1007,43.35
7,conditions Stormy,High,733,284,38.74
8,conditions Sandstorms,High,701,269,38.37
9,conditions Cloudy,High,744,280,37.63


### Finding

The combination of weather and traffic creates substantially higher delivery risk than either factor alone. **Jam + Fog** has an average delivery time of **36.81 minutes** and a **76.08% late-delivery rate**, while **Jam + Cloudy** has an average of **36.69 minutes** and a **74.84% late-delivery rate**.

**Business implication:** ETA planning and operational decisions should consider multiple environmental factors together rather than evaluating traffic and weather independently. High-risk combinations should trigger proactive intervention.

## 6. Distance Impact on Delivery Time

Analyze how delivery distance affects delivery time and late deliveries using SQL.

### Key Business Questions

- How does delivery distance affect average delivery time?
- Which distance range has the highest late-delivery rate?
- How does delivery performance change as distance increases?
- Are longer-distance deliveries more likely to be late?

In [ ]:
import numpy as np

lat1 = np.radians(df["Restaurant_latitude"])
lon1 = np.radians(df["Restaurant_longitude"])

lat2 = np.radians(df["Delivery_location_latitude"])
lon2 = np.radians(df["Delivery_location_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

df["Distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a))

In [22]:
df[["Distance_km"]].describe()

,Distance_km
count,41953.000000
mean,107.060101
std,1146.118637
min,1.465067
25%,4.658001
50%,9.220368
75%,13.760880
max,19692.674606


In [23]:
df.to_sql(
    "food_delivery",
    conn,
    if_exists="replace",
    index=False
)

print("SQL table updated successfully.")

SQL table updated successfully.


In [24]:
query = """
SELECT
    CASE
        WHEN "Distance_km" <= 5 THEN '0-5 km'
        WHEN "Distance_km" <= 10 THEN '5-10 km'
        WHEN "Distance_km" <= 15 THEN '10-15 km'
        WHEN "Distance_km" <= 20 THEN '15-20 km'
    END AS distance_range,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time
FROM food_delivery
WHERE "Distance_km" <= 20
GROUP BY distance_range
ORDER BY
    CASE distance_range
        WHEN '0-5 km' THEN 1
        WHEN '5-10 km' THEN 2
        WHEN '10-15 km' THEN 3
        WHEN '15-20 km' THEN 4
    END;
"""

pd.read_sql_query(query, conn)

,distance_range,total_deliveries,average_delivery_time
0,0-5 km,11187,22.18
1,5-10 km,11187,24.39
2,10-15 km,11183,29.84
3,15-20 km,6186,29.90


In [25]:
query = """
SELECT
    CASE
        WHEN "Distance_km" <= 5 THEN '0-5 km'
        WHEN "Distance_km" <= 10 THEN '5-10 km'
        WHEN "Distance_km" <= 15 THEN '10-15 km'
        WHEN "Distance_km" <= 20 THEN '15-20 km'
    END AS distance_range,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
WHERE "Distance_km" <= 20
GROUP BY distance_range
ORDER BY
    CASE distance_range
        WHEN '0-5 km' THEN 1
        WHEN '5-10 km' THEN 2
        WHEN '10-15 km' THEN 3
        WHEN '15-20 km' THEN 4
    END;
"""

pd.read_sql_query(query, conn)

,distance_range,total_deliveries,late_deliveries,late_delivery_rate
0,0-5 km,11187,1386,12.39
1,5-10 km,11187,2367,21.16
2,10-15 km,11183,5094,45.55
3,15-20 km,6186,2805,45.34


### Finding

For realistic delivery distances up to **20 km**, delivery time generally increases as distance increases. Average delivery time rises from **22.18 minutes** for 0–5 km deliveries to approximately **29.90 minutes** for 15–20 km deliveries.

The late-delivery rate also increases from **12.39%** for 0–5 km deliveries to approximately **45.34%** for 15–20 km deliveries.

**Business implication:** Delivery distance should be incorporated into ETA estimation and workload planning. Longer-distance orders require more realistic delivery-time expectations.

## 7. Vehicle Performance

Analyze how different vehicle types affect delivery time and late deliveries using SQL.

### Key Business Questions

- Which vehicle type has the lowest average delivery time?
- Which vehicle type has the highest late-delivery rate?
- How does delivery performance vary across vehicle types?
- Which vehicle type appears most efficient for deliveries?

In [29]:
query = """
SELECT
    "Type_of_vehicle" AS vehicle_type,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"),2) AS average_delivery_time,
    MIN("Time_taken(min)") AS minimum_delivery_time,
    MAX("Time_taken(min)") AS maximum_delivery_time
FROM food_delivery
GROUP BY "Type_of_vehicle"
ORDER BY average_delivery_time ASC;
"""
pd.read_sql_query(query,conn)
    

,vehicle_type,total_deliveries,average_delivery_time,minimum_delivery_time,maximum_delivery_time
0,electric_scooter,3814,24.47,10,53
1,scooter,15276,24.48,10,54
2,bicycle,68,26.43,10,47
3,motorcycle,26435,27.61,10,54


In [30]:
query = """
SELECT
    "Type_of_vehicle" AS vehicle_type,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
GROUP BY "Type_of_vehicle"
ORDER BY late_delivery_rate ASC;
"""

pd.read_sql_query(query, conn)

,vehicle_type,total_deliveries,late_deliveries,late_delivery_rate
0,electric_scooter,3814,894,23.44
1,scooter,15276,3718,24.34
2,bicycle,68,21,30.88
3,motorcycle,26435,8973,33.94


### Finding

Vehicle type is associated with differences in delivery performance. **Motorcycles** have the highest average delivery time at **27.61 minutes** and the highest late-delivery rate among the major vehicle categories at **33.94%**.

Scooters and electric scooters perform better, with average delivery times of approximately **24.48 and 24.47 minutes** and late-delivery rates below **25%**.

**Business implication:** Vehicle type should be considered when assigning deliveries, particularly for orders affected by traffic, distance, or workload constraints.

## 8. Delivery Partner Performance

Analyze delivery partner performance using delivery time and late-delivery rate to identify performance differences across partners.

### Key Business Questions

- Which delivery partners have the fastest average delivery times?
- Which delivery partners have the highest late-delivery rates?
- How much does delivery performance vary between partners?
- Which partners may require additional operational attention?

In [32]:
query = """
SELECT
    "Delivery_person_ID" AS delivery_partner,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time
FROM food_delivery
GROUP BY "Delivery_person_ID"
HAVING COUNT(*) >= 50
ORDER BY average_delivery_time ASC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,delivery_partner,total_deliveries,average_delivery_time
0,PUNERES07DEL02,59,22.71
1,MUMRES12DEL02,56,22.80
2,BANGRES15DEL02,59,23.07
3,HYDRES18DEL02,60,23.30
4,CHENRES08DEL02,54,23.59
5,CHENRES03DEL02,52,23.65
6,INDORES02DEL02,57,23.75
7,SURRES16DEL01,62,23.81
8,MUMRES17DEL02,56,23.84
9,COIMBRES01DEL02,55,24.00


In [33]:
query = """
SELECT
    "Delivery_person_ID" AS delivery_partner,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time
FROM food_delivery
GROUP BY "Delivery_person_ID"
HAVING COUNT(*) >= 50
ORDER BY average_delivery_time DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,delivery_partner,total_deliveries,average_delivery_time
0,CHENRES03DEL01,56,30.57
1,CHENRES14DEL01,52,30.10
2,PUNERES14DEL01,56,29.41
3,VADRES11DEL02,66,29.24
4,JAPRES010DEL01,55,29.15
5,COIMBRES16DEL02,58,29.09
6,CHENRES06DEL01,56,29.02
7,COIMBRES17DEL01,56,29.00
8,VADRES01DEL01,61,28.97
9,JAPRES09DEL03,61,28.95


In [34]:
query = """
SELECT
    "Delivery_person_ID" AS delivery_partner,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate,
    ROUND(AVG("Delivery_person_Ratings"), 2) AS average_rating
FROM food_delivery
GROUP BY "Delivery_person_ID"
HAVING COUNT(*) >= 50
ORDER BY late_delivery_rate ASC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,delivery_partner,total_deliveries,late_deliveries,late_delivery_rate,average_rating
0,PUNERES07DEL02,59,7,11.86,4.69
1,MUMRES12DEL02,56,7,12.50,4.69
2,HYDRES17DEL02,55,8,14.55,4.61
3,HYDRES18DEL02,60,9,15.00,4.71
4,INDORES12DEL01,61,10,16.39,4.67
5,BANGRES09DEL02,53,9,16.98,4.67
6,VADRES01DEL02,55,10,18.18,4.66
7,CHENRES08DEL02,54,10,18.52,4.55
8,MUMRES09DEL01,54,10,18.52,4.64
9,MYSRES20DEL02,59,11,18.64,4.70


In [35]:
query = """
SELECT
    "Delivery_person_ID" AS delivery_partner,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate,
    ROUND(AVG("Delivery_person_Ratings"), 2) AS average_rating
FROM food_delivery
GROUP BY "Delivery_person_ID"
HAVING COUNT(*) >= 50
ORDER BY late_delivery_rate DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,delivery_partner,total_deliveries,late_deliveries,late_delivery_rate,average_rating
0,CHENRES14DEL01,52,29,55.77,4.59
1,CHENRES03DEL01,56,28,50.00,4.55
2,JAPRES010DEL01,55,27,49.09,4.61
3,COIMBRES05DEL02,53,25,47.17,4.62
4,CHENRES17DEL01,50,23,46.00,4.52
5,INDORES19DEL01,57,26,45.61,4.54
6,SURRES010DEL01,58,26,44.83,4.61
7,COIMBRES16DEL01,56,25,44.64,4.62
8,RANCHIRES13DEL01,62,27,43.55,4.57
9,SURRES09DEL02,58,25,43.10,4.51


### Finding

Delivery partner performance varies substantially. Among partners with at least **50 deliveries**, the fastest-performing partner, **PUNERES07DEL02**, has an average delivery time of **22.71 minutes**, while **CHENRES03DEL01** has the highest average at **30.57 minutes**.

This represents a difference of approximately **7.86 minutes** in average delivery time.

**Business implication:** Delivery partner performance should be monitored regularly. High-performing partners can provide operational benchmarks, while consistently slower performance may require targeted training or workload review.

## 9. Multiple Deliveries

Analyze how handling multiple deliveries affects delivery time and late-delivery rates.

### Key Business Questions

- How does the number of multiple deliveries affect average delivery time?
- Which multiple-delivery category has the highest late-delivery rate?
- Does handling multiple deliveries increase the risk of delays?
- What operational impact does multiple delivery assignment have?

In [36]:
query = """
SELECT
    "multiple_deliveries" AS multiple_deliveries,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time
FROM food_delivery
GROUP BY "multiple_deliveries"
ORDER BY average_delivery_time DESC;
"""

pd.read_sql_query(query, conn)

,multiple_deliveries,total_deliveries,average_delivery_time
0,3,361,47.82
1,2,1985,40.45
2,1,28159,26.86
3,0,14095,22.88
4,NaN,993,22.77


In [37]:
query = """
SELECT
    "multiple_deliveries" AS multiple_deliveries,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
GROUP BY "multiple_deliveries"
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,multiple_deliveries,total_deliveries,late_deliveries,late_delivery_rate
0,3,361,361,100.00
1,2,1985,1985,100.00
2,1,28159,8521,30.26
3,NaN,993,185,18.63
4,0,14095,2554,18.12


### Finding

Delivery workload has a strong relationship with delivery time. Orders involving **2 deliveries** have an average delivery time of **40.45 minutes**, while those involving **3 deliveries** average **47.82 minutes**. Both groups have a historical **100% late-delivery rate**.

In comparison, orders involving 0 deliveries average **22.88 minutes** with an **18.12% late-delivery rate**.

**Business implication:** Multiple-order assignments should be carefully controlled, particularly during difficult operating conditions such as heavy traffic or poor weather.

## 10. City-Level Performance

Analyze delivery performance across cities to identify geographic differences in delivery time and delays.

### Key Business Questions

- Which city has the highest average delivery time?
- Which city has the highest late-delivery rate?
- How does delivery performance vary across cities?
- Which cities may require operational improvements?

In [38]:
query = """
SELECT
    "City" AS city,
    COUNT(*) AS total_deliveries,
    ROUND(AVG("Time_taken(min)"), 2) AS average_delivery_time,
    MIN("Time_taken(min)") AS minimum_delivery_time,
    MAX("Time_taken(min)") AS maximum_delivery_time
FROM food_delivery
GROUP BY "City"
ORDER BY average_delivery_time DESC;
"""

pd.read_sql_query(query, conn)

,city,total_deliveries,average_delivery_time,minimum_delivery_time,maximum_delivery_time
0,Semi-Urban,164,49.73,44,54
1,Metropolitian,34093,27.32,10,54
2,Urban,10136,22.98,10,54
3,NaN,1200,22.06,10,54


In [39]:
query = """
SELECT
    "City" AS city,
    COUNT(*) AS total_deliveries,
    SUM(
        CASE
            WHEN "Time_taken(min)" > 30 THEN 1
            ELSE 0
        END
    ) AS late_deliveries,
    ROUND(
        SUM(
            CASE
                WHEN "Time_taken(min)" > 30 THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_rate
FROM food_delivery
GROUP BY "City"
ORDER BY late_delivery_rate DESC;
"""

pd.read_sql_query(query, conn)

,city,total_deliveries,late_deliveries,late_delivery_rate
0,Semi-Urban,164,164,100.00
1,Metropolitian,34093,11366,33.34
2,Urban,10136,1891,18.66
3,NaN,1200,185,15.42


### Finding

Delivery performance varies across city types. **Metropolitian** areas have an average delivery time of **27.32 minutes** and a **33.34% late-delivery rate**, while **Urban** areas average **22.98 minutes** with an **18.66% late-delivery rate**.

The Semi-Urban category shows a very high average delivery time and 100% late rate, but it contains only **164 orders**, so this result should be interpreted cautiously.

**Business implication:** City type can be considered when planning delivery capacity and ETAs, but small sample groups should not drive major operational decisions without further validation.

## 11. Outlier Analysis

Identify unusually high delivery times using SQL to understand the extent of extreme delivery delays.

### Key Business Questions

- What are the unusually high delivery times in the dataset?
- How many deliveries are potential outliers?
- What percentage of deliveries are potential outliers?
- Should extreme delivery times be investigated or removed?

In [40]:
query = """
WITH quartiles AS (
    SELECT
        "Time_taken(min)" AS delivery_time,
        NTILE(4) OVER (ORDER BY "Time_taken(min)") AS quartile
    FROM food_delivery
),
q1_q3 AS (
    SELECT
        MAX(CASE WHEN quartile = 1 THEN delivery_time END) AS q1,
        MAX(CASE WHEN quartile = 3 THEN delivery_time END) AS q3
    FROM quartiles
)
SELECT
    q1,
    q3,
    ROUND(q3 - q1, 2) AS iqr,
    ROUND(q3 + 1.5 * (q3 - q1), 2) AS upper_bound
FROM q1_q3;
"""

pd.read_sql_query(query, conn)

,q1,q3,iqr,upper_bound
0,19,32,13.0,51.5


In [41]:
query = """
WITH quartiles AS (
    SELECT
        "Time_taken(min)" AS delivery_time,
        NTILE(4) OVER (ORDER BY "Time_taken(min)") AS quartile
    FROM food_delivery
),
q1_q3 AS (
    SELECT
        MAX(CASE WHEN quartile = 1 THEN delivery_time END) AS q1,
        MAX(CASE WHEN quartile = 3 THEN delivery_time END) AS q3
    FROM quartiles
)
SELECT
    COUNT(*) AS total_outliers,
    ROUND(
        COUNT(*) * 100.0 / (SELECT COUNT(*) FROM food_delivery),
        2
    ) AS outlier_percentage
FROM food_delivery
CROSS JOIN q1_q3
WHERE "Time_taken(min)" > q3 + 1.5 * (q3 - q1);
"""

pd.read_sql_query(query, conn)

,total_outliers,outlier_percentage
0,270,0.59


### Finding

Using the IQR method, the upper outlier threshold for delivery time is **51.5 minutes**. The analysis identifies **270 outlier deliveries**, representing approximately **0.59%** of all deliveries.

These observations are relatively small in number but may represent unusual operational situations.

**Business implication:** Outliers should be investigated rather than automatically removed, as they may reveal specific operational problems or exceptional delivery conditions.

## 12. Correlation Analysis

Analyze relationships between numerical operational factors and delivery time using SQL.

### Key Business Questions

- Which factors are most strongly associated with delivery time?
- Does the number of multiple deliveries increase delivery time?
- How are delivery partner ratings related to delivery time?M
- Which operational factors should be investigated further?`M

In [43]:
query = """
SELECT
    'multiple_deliveries' AS factor,
    ROUND(
        (
            COUNT(*) * SUM("multiple_deliveries" * "Time_taken(min)")
            - SUM("multiple_deliveries") * SUM("Time_taken(min)")
        )
        /
        SQRT(
            (
                COUNT(*) * SUM("multiple_deliveries" * "multiple_deliveries")
                - SUM("multiple_deliveries") * SUM("multiple_deliveries")
            )
            *
            (
                COUNT(*) * SUM("Time_taken(min)" * "Time_taken(min)")
                - SUM("Time_taken(min)") * SUM("Time_taken(min)")
            )
        ),
        3
    ) AS correlation
FROM food_delivery
WHERE "multiple_deliveries" IS NOT NULL

UNION ALL

SELECT
    'Delivery_person_Age' AS factor,
    ROUND(
        (
            COUNT(*) * SUM("Delivery_person_Age" * "Time_taken(min)")
            - SUM("Delivery_person_Age") * SUM("Time_taken(min)")
        )
        /
        SQRT(
            (
                COUNT(*) * SUM("Delivery_person_Age" * "Delivery_person_Age")
                - SUM("Delivery_person_Age") * SUM("Delivery_person_Age")
            )
            *
            (
                COUNT(*) * SUM("Time_taken(min)" * "Time_taken(min)")
                - SUM("Time_taken(min)") * SUM("Time_taken(min)")
            )
        ),
        3
    ) AS correlation
FROM food_delivery
WHERE "Delivery_person_Age" IS NOT NULL;
"""

pd.read_sql_query(query, conn)

,factor,correlation
0,multiple_deliveries,0.386
1,Delivery_person_Age,0.300


In [44]:
query = """
SELECT
    'Vehicle_condition' AS factor,
    ROUND(
        (
            COUNT(*) * SUM("Vehicle_condition" * "Time_taken(min)")
            - SUM("Vehicle_condition") * SUM("Time_taken(min)")
        )
        /
        SQRT(
            (
                COUNT(*) * SUM("Vehicle_condition" * "Vehicle_condition")
                - SUM("Vehicle_condition") * SUM("Vehicle_condition")
            )
            *
            (
                COUNT(*) * SUM("Time_taken(min)" * "Time_taken(min)")
                - SUM("Time_taken(min)") * SUM("Time_taken(min)")
            )
        ),
        3
    ) AS correlation
FROM food_delivery

UNION ALL

SELECT
    'Delivery_person_Ratings' AS factor,
    ROUND(
        (
            COUNT(*) * SUM("Delivery_person_Ratings" * "Time_taken(min)")
            - SUM("Delivery_person_Ratings") * SUM("Time_taken(min)")
        )
        /
        SQRT(
            (
                COUNT(*) * SUM("Delivery_person_Ratings" * "Delivery_person_Ratings")
                - SUM("Delivery_person_Ratings") * SUM("Delivery_person_Ratings")
            )
            *
            (
                COUNT(*) * SUM("Time_taken(min)" * "Time_taken(min)")
                - SUM("Time_taken(min)") * SUM("Time_taken(min)")
            )
        ),
        3
    ) AS correlation
FROM food_delivery
WHERE "Delivery_person_Ratings" IS NOT NULL;
"""

pd.read_sql_query(query, conn)

,factor,correlation
0,Vehicle_condition,-0.234
1,Delivery_person_Ratings,-0.341


### Finding

The correlation analysis shows that **multiple deliveries** have the strongest positive relationship with delivery time, with a correlation of approximately **0.386**. Delivery partner age also has a positive relationship (**0.300**).

Delivery partner ratings have a moderate negative relationship (**-0.341**), while vehicle condition has a negative relationship (**-0.234**).

**Business implication:** Workload, delivery partner characteristics, and vehicle condition may be useful inputs for operational planning and delivery-time prediction. However, correlation indicates association and does not establish causation.